# s27 — FSM Communication Protocol

**What this teaches:** how the `teammates.Agent` value encodes its communication protocol as an explicit finite-state machine. We bypass the LLM here and drive two agents (`lead`, `reviewer`) directly so the state transitions — `IDLE → REQUESTING → WAITING → RESPONDING → IDLE` — become observable.

**Concept anchor:** the FSM is what gives `teammate_ask` its ordering guarantee. An agent can only have **one outstanding question at a time**; the FSM blocks `Ask` until any previous reply lands, so concurrent conversations never interleave their request/reply pairs. The same FSM runs underneath the LLM-facing tools used in [s26_mailbox](../s26_mailbox/s26_mailbox.ipynb).

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- **No LLM required.** This example only uses `internal/teammates` directly, so you can run it without setting any provider env vars.

## 1. Imports

We import `teammates` for the FSM-backed agents and `time` to set ask/reply deadlines.

In [ ]:
import (
    "context"
    "fmt"
    "time"

    "github.com/blouargant/yoke/internal/teammates"
)

## 2. Helper

Notebook-safe `must` — panics instead of `os.Exit`.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Two agents on a shared backend

Both agents bind to the **same** mailbox backend, so they can reach each other by name. In production the backend is usually Redis (see [s29_redis](../s29_redis/s29_redis.ipynb)); here the in-memory default is enough.

In [ ]:
be, err := teammates.ChooseBackend()
must(err)
defer be.Close()
lead := teammates.NewAgent("lead", be)
rev := teammates.NewAgent("reviewer", be)
fmt.Println("lead initial state:", lead.State())
fmt.Println("reviewer initial state:", rev.State())

## 4. Park the reviewer in RESPONDING

`rev.Check(ctx, timeout)` blocks until a message arrives. While blocked the FSM is in `WAITING`; once the message is delivered the FSM moves to `RESPONDING` until the reply is sent. We launch the reviewer in a goroutine and busy-wait briefly so the lead's `Ask` lands during `WAITING`.

In [ ]:
ctx, cancel := context.WithTimeout(context.Background(), 5*time.Second)
defer cancel()
go func() {
    m, err := rev.Check(ctx, 4*time.Second)
    if err != nil || m == nil {
        return
    }
    _ = rev.Tell(ctx, m.From, "LGTM")
}()

deadline := time.Now().Add(200 * time.Millisecond)
for time.Now().Before(deadline) && rev.State() != teammates.StateResponding {
    time.Sleep(5 * time.Millisecond)
}
fmt.Println("reviewer state before ask:", rev.State())

## 5. Lead asks, FSM transitions

`lead.Ask` drives `IDLE → REQUESTING → WAITING`, then back to `IDLE` once the reply arrives. The FSM enforces a single outstanding question at a time: a second concurrent `Ask` from the same lead would block until this one completes. That ordering guarantee is what makes ask/reply safe to use across long-running conversations.

In [ ]:
fmt.Printf("lead state before ask: %s\n", lead.State())
reply, err := lead.Ask(ctx, "reviewer", "ok to merge?", 3*time.Second)
must(err)
fmt.Printf("lead state after  ask: %s\n", lead.State())
fmt.Println("reply:", reply)
fmt.Println("reviewer final state:", rev.State())

## What to look for

- The printed transitions tell the story: reviewer briefly in `RESPONDING`, lead drops back to `IDLE` once the reply is received. Compare with [s26_mailbox](../s26_mailbox/s26_mailbox.ipynb), where the same FSM is hidden behind LLM tools.
- The `Ask`/`Check`/`Tell` API is **safe to call concurrently across distinct agents** because each `Agent` owns its FSM. Two leads can `Ask` the same reviewer in parallel; the reviewer's `Check` will serialize them.
- For a distributed take on the same FSM, see [s29_redis](../s29_redis/s29_redis.ipynb) — the state machine doesn't change, only the backend.

## Try it yourself

1. Remove the busy-wait loop and observe the lead's `Ask` landing before the reviewer enters `RESPONDING`. The FSM still resolves correctly — it just goes through `WAITING` for a moment.
2. Add a second `lead.Ask` immediately after the first (in a goroutine) and watch the FSM serialize them — the second one parks in `REQUESTING` until the first completes.